# pLaplace FEniCS Benchmark

This notebook demonstrates the current maintained CLI commands on lightweight benchmark cases.

Relevant docs:
- [docs/problems/pLaplace.md](../../docs/problems/pLaplace.md)
- [docs/results/pLaplace.md](../../docs/results/pLaplace.md)


In [1]:
from pathlib import Path
import json
import subprocess

REPO_ROOT = Path.cwd()
ARTIFACT_ROOT = Path('artifacts/raw_results/notebook_runs')
(REPO_ROOT / ARTIFACT_ROOT).mkdir(parents=True, exist_ok=True)
PYTHON = './.venv/bin/python'

def dump_json(path):
    with open(REPO_ROOT / path, encoding='utf-8') as handle:
        return json.load(handle)


In [2]:
from pprint import pprint

out_dir = ARTIFACT_ROOT / 'plaplace_fenics_benchmark'
(REPO_ROOT / out_dir).mkdir(parents=True, exist_ok=True)

def run_case(name, argv):
    out_path = out_dir / f'{name}.json'
    cmd = argv + [str(out_path)]
    print('$', ' '.join(cmd))
    subprocess.run(cmd, cwd=REPO_ROOT, check=True)
    payload = dump_json(out_path)
    row = payload['results'][0]
    return {
        'case': name,
        'dofs': row.get('dofs', row.get('total_dofs')),
        'iters': row.get('iters'),
        'energy': row.get('energy'),
        'time': row.get('time', row.get('solve_time', row.get('total_time'))),
    }

results = [
    run_case('custom_serial', [PYTHON, 'src/problems/plaplace/fenics/solve_pLaplace_custom_jaxversion.py', '--levels', '5', '--quiet', '--json']),
    run_case('snes_serial', [PYTHON, 'src/problems/plaplace/fenics/solve_pLaplace_snes_newton.py', '--levels', '5', '--json']),
    run_case('custom_mpi2', ['mpiexec', '-n', '2', PYTHON, 'src/problems/plaplace/fenics/solve_pLaplace_custom_jaxversion.py', '--levels', '5', '--quiet', '--json']),
]
pprint(results)


$ ./.venv/bin/python src/problems/plaplace/fenics/solve_pLaplace_custom_jaxversion.py --levels 5 --quiet --json artifacts/raw_results/notebook_runs/plaplace_fenics_benchmark/custom_serial.json


p-Laplace 2D Custom Newton (JAX-version) | 1 MPI process(es)
  --- Mesh level 5 ---
  RESULT mesh_level=5 dofs=3201 setup=0.015s solve=0.079s iters=5 ksp=15 asm=0.006s J(u)=-7.942969 [Converged (energy, step, gradient)]
Done.
Results saved to artifacts/raw_results/notebook_runs/plaplace_fenics_benchmark/custom_serial.json


$ ./.venv/bin/python src/problems/plaplace/fenics/solve_pLaplace_snes_newton.py --levels 5 --json artifacts/raw_results/notebook_runs/plaplace_fenics_benchmark/snes_serial.json


p-Laplace 2D SNES Newton | 1 MPI process(es)
  mesh_level=5 dofs=3201 time=0.070s iters=10 J(u)=-7.9430 reason=2
Done.
Results saved to artifacts/raw_results/notebook_runs/plaplace_fenics_benchmark/snes_serial.json


$ mpiexec -n 2 ./.venv/bin/python src/problems/plaplace/fenics/solve_pLaplace_custom_jaxversion.py --levels 5 --quiet --json artifacts/raw_results/notebook_runs/plaplace_fenics_benchmark/custom_mpi2.json


p-Laplace 2D Custom Newton (JAX-version) | 2 MPI process(es)
  --- Mesh level 5 ---
  RESULT mesh_level=5 dofs=3201 setup=0.021s solve=0.061s iters=5 ksp=15 asm=0.004s J(u)=-7.942969 [Converged (energy, step, gradient)]
Done.
Results saved to artifacts/raw_results/notebook_runs/plaplace_fenics_benchmark/custom_mpi2.json


[{'case': 'custom_serial',
  'dofs': 3201,
  'energy': -7.9429687187,
  'iters': 5,
  'time': 0.0787},
 {'case': 'snes_serial',
  'dofs': 3201,
  'energy': -7.943,
  'iters': 10,
  'time': 0.07},
 {'case': 'custom_mpi2',
  'dofs': 3201,
  'energy': -7.9429687182,
  'iters': 5,
  'time': 0.0608}]


Canonical commands shown above write notebook outputs under `artifacts/raw_results/notebook_runs/plaplace_fenics_benchmark/`.
